# Offline Showdex -> poke-engine MCTS -> Policy MLP Training

This notebook runs the full offline policy-predictor workflow: open or clone `showdown-trainer`, install dependencies, download the same pkmn preset data paths used by Showdex, collect Showdex-guided `poke-engine` MCTS targets into JSONL, run sanity checks, train the policy MLP, and verify the checkpoint.

No Pokemon Showdown login or websocket battle is required. After the Showdex/pkmn JSON files are cached locally, collection is offline-capable.

## 1. Locate Or Clone The Trainer Repo

If this notebook is already inside a local clone, the setup cell reuses it. In Colab, set `TRAINER_REPO_URL` to your fork before running.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

TRAINER_REPO_URL = "https://github.com/YOUR_USERNAME/showdown-trainer.git"  # edit for Colab
BRANCH = None  # e.g. "main" or your feature branch

def run(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    return subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)

def looks_like_repo(path: Path) -> bool:
    return (path / "train.py").exists() and (path / "collect_offline_mcts.py").exists()

cwd = Path.cwd()
base_dir = Path("/content") if "google.colab" in sys.modules else cwd

if looks_like_repo(cwd):
    repo = cwd
else:
    repo = base_dir / "showdown-trainer"
    if not repo.exists():
        clone_cmd = ["git", "clone"]
        if BRANCH:
            clone_cmd += ["--branch", BRANCH]
        clone_cmd += [TRAINER_REPO_URL, repo]
        run(clone_cmd)

os.chdir(repo)
print(f"Using repo: {repo.resolve()}")


## 2. Install Python Dependencies

`poke-engine` provides the offline MCTS search. Colab already has PyTorch in most runtimes, but installing from `requirements.txt` keeps the notebook self-contained.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import poke_engine
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("poke_engine loaded")


## 3. Cache Showdex / pkmn Distributions

Showdex uses the pkmn repository under `https://pkmn.github.io`. This cell downloads the random battle preset and usage-stat JSON once. Reruns use the cached local files in `showdex_cache/`.

In [ ]:
import urllib.request

POKEMON_FORMAT = "gen9randombattle"
SHOWDEX_CACHE = Path("showdex_cache")
SHOWDEX_CACHE.mkdir(parents=True, exist_ok=True)

downloads = {
    SHOWDEX_CACHE / f"{POKEMON_FORMAT}.json": f"https://pkmn.github.io/randbats/data/{POKEMON_FORMAT}.json",
    SHOWDEX_CACHE / f"{POKEMON_FORMAT}-stats.json": f"https://pkmn.github.io/randbats/data/stats/{POKEMON_FORMAT}.json",
}

for path, url in downloads.items():
    if path.exists() and path.stat().st_size > 0:
        print(f"Using cached {path}")
        continue
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, path)
    print(f"Wrote {path} ({path.stat().st_size:,} bytes)")


## 4. Collect Offline MCTS Training Data

For a quick notebook verification, `SMOKE_RUN=True` keeps the search small. Increase `POSITIONS`, `HYPOTHESES`, and `MCTS_MS` for a larger training set.

In [ ]:
CONFIG_PATH = Path("configs/student.yaml")
DATA_PATH = Path("training_data/mcts_run.jsonl")
CHECKPOINT_PATH = Path("checkpoints/policy_mlp.pt")

SMOKE_RUN = True
POSITIONS = 8 if SMOKE_RUN else 64
HYPOTHESES = 2 if SMOKE_RUN else 4
MCTS_MS = 25 if SMOKE_RUN else 75

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if DATA_PATH.exists():
    DATA_PATH.unlink()

run([
    sys.executable,
    "collect_offline_mcts.py",
    "--config", CONFIG_PATH,
    "--positions", POSITIONS,
    "--hypotheses", HYPOTHESES,
    "--search-time-ms", MCTS_MS,
])

print(DATA_PATH, DATA_PATH.stat().st_size, "bytes")
print("rows", sum(1 for _ in DATA_PATH.open()))


## 5. Run Sanity Checks

In [ ]:
run([sys.executable, "sanity_check.py"])


## 6. Train The Policy MLP

The training script saves model and optimizer state, so the checkpoint can be resumed later with `training.resume_checkpoint_path` or `--resume-checkpoint-path`.

In [ ]:
train_cmd = [sys.executable, "train.py", "--config", CONFIG_PATH]
if SMOKE_RUN:
    train_cmd.append("--smoke")
run(train_cmd)


## 7. Verify The Checkpoint

In [ ]:
import json

from encode import EncoderConfig
from train import load_policy_checkpoint, predict_policy

model, checkpoint = load_policy_checkpoint(CHECKPOINT_PATH)
assert "model_state_dict" in checkpoint
assert "optimizer_state_dict" in checkpoint

first_decision = None
with DATA_PATH.open() as file:
    for line in file:
        record = json.loads(line)
        if record.get("record_type") == "decision":
            first_decision = record
            break

assert first_decision is not None
encoder_config = EncoderConfig(**checkpoint["encoder_config"])
probs = predict_policy(model, first_decision["state"], encoder_config=encoder_config)

print("checkpoint", CHECKPOINT_PATH.resolve())
print("saved update", checkpoint.get("update"), "saved epoch", checkpoint.get("epoch"))
print("prob sum", round(sum(probs), 6))
print("policy probs", [round(value, 4) for value in probs])


## 8. Scaling Up

For a larger Colab run, set `SMOKE_RUN=False`, then raise collection gradually: `POSITIONS=256`, `HYPOTHESES=4`, `MCTS_MS=75` is a reasonable next step. Resume training by setting `training.resume_checkpoint_path: checkpoints/policy_mlp.pt` in `configs/student.yaml` or by passing `--resume-checkpoint-path checkpoints/policy_mlp.pt` to `train.py`.